# Lesson 77 · Music features — hearing a song's "key fingerprint": chroma, RMS energy and zero-crossing rate (ZCR) from scratch

**Learning goals**
1. Understand how a chroma vector folds a spectrum by pitch class.
2. Implement frame-by-frame RMS energy envelopes and ZCR.
3. Understand the basic idea behind autocorrelation beat detection.
4. Be able to match these three kinds of feature against the reference implementations in `aurora.music`.

> **Feature table**: name — meaning — dimension; compared with MFCC (timbre), chroma leans toward key.

🔗 **Aurora connection**: the `aurora.music` sub-package is fully implemented in `src/aurora/music/features.py` and exports `chroma_vector`, `rms_envelope`, `zero_crossing_rate`, `onset_envelope` and `beat_track`. The exercises in this section write these functions from scratch precisely so you understand how they work inside; when you're done you can compare them line by line with the reference implementation in `src/aurora/music/features.py`.

← **Previous lesson** · [L76 · Music theory crash course](L76_music_theory.ipynb)

> Last lesson covered **a music theory crash course**: pitch, intervals, the chroma wheel and equal temperament.
> This lesson explores **music features**.

## This lesson's story: how does an AI "hear" the key of a song?

Play someone the first couple of notes of "Twinkle, Twinkle, Little Star" and they can tell it's in C major. What is behind that "intuition"?

**A sense of key = which pitch classes carry the most energy.**

In a piece in C major, C, E and G (the tonic triad) carry the most audio energy — group the FFT power spectrum by pitch class (chroma) and add it up, and you get this "key fingerprint": **the chroma vector, 12-dimensional**.

**Three features, three questions**:

| Feature | What question it answers | Shape |
|---|---|---|
| chroma_vector | "What key is this music in? What chords does it use?" | (12,) |
| rms_envelope | "How does loudness change over time? Where are the strong beats?" | (n_frames,) |
| zero_crossing_rate | "Is this noise (an instrument's transition sound) or a musical tone?" | (n_frames,) |

**How chroma is computed**:
```
FFT power spectrum (n_fft//2+1,)
   ↓ frequency of each bin f = k × sr/n_fft
   ↓ frequency → pitch class c = round(12×log2(f/440) + 69) % 12
   ↓ sum the power for c = 0..11
   ↓ normalize (divide by the sum or the maximum)
   → chroma vector (12,)
```

**RMS** is a "short-time loudness meter": `sqrt(mean(x^2))` per frame; its time resolution depends on the hop size.
**ZCR** counts how often the signal crosses zero: musical tones have low ZCR (stable frequency), noise has high ZCR (random crossings).

Your task in this section: implement these three feature functions from scratch, and check them numerically with pure tones and standard signals.

In [ ]:
# Aurora matplotlib bootstrap
from pathlib import Path
import sys

_root = None
_cwd = Path.cwd().resolve()
for _candidate in (_cwd, *_cwd.parents):
    if (_candidate / '_matplotlib_bootstrap.py').exists():
        _root = _candidate
        break
if _root is None:
    _notebooks_dir = _cwd / 'notebooks'
    if _notebooks_dir.exists():
        for _found in _notebooks_dir.rglob('_matplotlib_bootstrap.py'):
            _root = _found.parent
            break
if _root is not None and str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from _matplotlib_bootstrap import apply as _aurora_mpl_apply
_aurora_mpl_apply()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 1. Chroma: the key fingerprint

The STFT gives the frequency of each FFT bin, `f = k * sr / n_fft` (k = 0,1,...,n_fft//2). Map each frequency to a MIDI note:

```
midi = 12 * log2(f / 440) + 69
```

Pitch class = `round(midi) mod 12`, with C=0, C#=1, D=2, ..., B=11. Add up the power of every bin that lands in the same pitch class, giving a length-12 vector, then normalize it to [0,1].

Intuition: when an A major scale (A B C# D E F# G#) is played, chroma positions 1, 2, 4, 6, 8, 9, 11 should stand well above the rest.

In [ ]:
# Demo: build a pure tone containing only A4 (440 Hz) and look at its chroma
sr = 22050
duration = 1.0
t = np.arange(int(sr * duration)) / sr
x_pure_a = np.sin(2 * np.pi * 440 * t)   # pure A4

n_fft = 2048
freqs = np.arange(n_fft // 2 + 1) * sr / n_fft  # frequency of each bin

# compute the power spectrum by hand (single-frame demo)
frame = x_pure_a[:n_fft] * np.hanning(n_fft)
spectrum = np.abs(np.fft.rfft(frame)) ** 2

# pitch-class mapping
with np.errstate(divide='ignore', invalid='ignore'):
    midi = np.where(freqs > 0, 12 * np.log2(freqs / 440) + 69, -1)
pitch_class = (np.round(midi).astype(int)) % 12

chroma_demo = np.zeros(12)
for pc, p in zip(pitch_class, spectrum):
    if 0 <= pc < 12:
        chroma_demo[pc] += p
chroma_demo /= (chroma_demo.max() + 1e-8)

labels = ['C','C#','D','D#','E','F','F#','G','G#','A','A#','B']
plt.figure(figsize=(8, 3))
plt.bar(labels, chroma_demo)
plt.title('Chroma — pure A4 (440 Hz); expect A=9 to be highest')
plt.ylabel('Normalized energy')
plt.tight_layout()
plt.show()
print(f'Pitch class with the most energy: index = {chroma_demo.argmax()}  (should be 9 = A)')

## 2. RMS energy envelope

Split the audio signal into short frames (frame length `frame_len`, hop `hop`) and compute the root mean square of each frame:

```
rms[i] = sqrt( mean( frame_i ** 2 ) )
```

The result is a vector of length `n_frames` describing how loudness changes over time. Silent passages have RMS ≈ 0; drum hits or full-ensemble passages can reach RMS 0.3–0.8 (for audio normalized to ±1).

Frame-count formula: `n_frames = 1 + (len(x) - frame_len) // hop`.

In [ ]:
# Demo: square wave spliced between silences; look at the RMS envelope
sr = 22050
silence = np.zeros(sr // 2)            # 0.5 s of silence
loud = 0.8 * np.sign(np.sin(2 * np.pi * 220 * np.arange(sr) / sr))  # 1 s square wave
x_demo = np.concatenate([silence, loud, silence])

frame_len, hop = 2048, 512
n_frames = 1 + (len(x_demo) - frame_len) // hop
frames = np.array([x_demo[i*hop : i*hop + frame_len] for i in range(n_frames)])
rms_demo = np.sqrt(np.mean(frames ** 2, axis=1))

times = np.arange(n_frames) * hop / sr
plt.figure(figsize=(9, 3))
plt.plot(times, rms_demo)
plt.xlabel('Time (s)')
plt.ylabel('RMS')
plt.title('RMS energy envelope: silence → square wave → silence')
plt.tight_layout()
plt.show()

## 3. Beat detection (the autocorrelation method)

**The idea**:
1. Compute the RMS energy envelope (a rough stand-in for onset strength).
2. Autocorrelate the envelope: `ac[lag] = sum( rms[t] * rms[t+lag] )`.
3. Within the lag range that corresponds to plausible BPMs (usually 60–240 BPM), find the peak lag*.
4. `BPM = 60 / (lag* * hop / sr)`.

```
lag_min = int(60 / 240 * sr / hop)   # corresponds to 240 BPM
lag_max = int(60 / 60  * sr / hop)   # corresponds to  60 BPM
lag_star = lag_min + ac[lag_min:lag_max].argmax()
BPM = 60 / (lag_star * hop / sr)
```

As long as there is a periodic beat, the autocorrelation sprouts a clear peak at the matching lag. The method needs no labeled data — it is an unsupervised estimate.

In [ ]:
# Demo: build a 120 BPM drum-hit envelope and test the autocorrelation BPM estimate
sr = 22050
hop = 512
frame_len = 2048
bpm_true = 120
beat_period_samples = int(60 / bpm_true * sr)   # samples per beat

# simulate drum transients with decaying bursts
x_drum = np.zeros(sr * 4)   # 4 s
for beat_start in range(0, len(x_drum), beat_period_samples):
    end = min(beat_start + 2205, len(x_drum))  # 100 ms decay
    length = end - beat_start
    x_drum[beat_start:end] += np.exp(-np.arange(length) / 1000) * np.random.randn(length) * 0.5

# RMS envelope
n_frames = 1 + (len(x_drum) - frame_len) // hop
frames_d = np.array([x_drum[i*hop:i*hop+frame_len] for i in range(n_frames)])
rms_d = np.sqrt(np.mean(frames_d**2, axis=1))

# autocorrelation
ac = np.correlate(rms_d, rms_d, mode='full')
ac = ac[len(ac)//2:]   # keep positive lags only

lag_min = max(1, int(60/240 * sr / hop))
lag_max = int(60/60  * sr / hop)
lag_star = lag_min + ac[lag_min:lag_max].argmax()
bpm_est = 60 / (lag_star * hop / sr)

print(f'True BPM = {bpm_true},  estimated BPM = {bpm_est:.1f}')

plt.figure(figsize=(9,3))
lags = np.arange(lag_min, lag_max) * hop / sr
plt.plot(lags, ac[lag_min:lag_max])
plt.axvline(lag_star * hop / sr, color='r', linestyle='--', label=f'peak lag → {bpm_est:.1f} BPM')
plt.xlabel('Lag (s)')
plt.ylabel('Autocorrelation')
plt.title('Autocorrelation of the RMS envelope — the peak marks the beat period')
plt.legend()
plt.tight_layout()
plt.show()

## 4. ✏️ Implement `chroma_vector(power_spectrum, sr, n_fft)`

**Signature**:
```python
def chroma_vector(power_spectrum: np.ndarray, sr: int, n_fft: int) -> np.ndarray:
    # power_spectrum: (n_fft//2+1,) power spectrum (magnitude squared)
    # returns: (12,) float, the chroma vector, normalized (divided by its sum; all zeros if the input is all zeros)
```

**A 4-step route to the implementation**:

| Step | Operation | Formula |
|---|---|---|
| 1 | Compute the frequency of each FFT bin | `f[k] = k × sr / n_fft`, k=0..n_fft//2 |
| 2 | Frequency → pitch class (skip f=0; its logarithm is meaningless) | `c = round(12×log2(f/440) + 69) % 12` |
| 3 | Sum the power by chroma | `chroma[c] += power_spectrum[k]` |
| 4 | Normalize | `chroma /= chroma.sum()` (if sum==0, leave it all zeros) |

> The constant **69** is the MIDI number of A4=440Hz (69 = 5×12 + 9); after mod 12 it is equivalent to writing `+9` directly — this lesson uses +69 throughout, to match the introductory formula and the reference implementation.

**Acceptance criteria**:
- Pure A4 (440 Hz) signal: `chroma_vector(...)[9]` is the maximum (A = chroma 9)
- Output shape=(12,), all values non-negative, sum≈1.0 (or all zeros)

In [ ]:
def chroma_vector(power_spectrum, sr, n_fft):
    """
    Map a power spectrum to a 12-dimensional chroma vector.

    Parameters
    ----------
    power_spectrum : np.ndarray, shape (n_fft//2 + 1,)
    sr : int  sample rate
    n_fft : int  number of FFT points

    Returns
    -------
    chroma : np.ndarray, shape (12,)  normalized (divided by the sum; all zeros if the input is all zeros)
    """
    raise NotImplementedError("chroma_vector: please implement — follow the 4-step route")
    n_bins = len(power_spectrum)
    freqs = np.arange(n_bins) * sr / n_fft  # ✏️ TODO: frequency of each bin

    chroma = np.zeros(12)
    for k, (f, p) in enumerate(zip(freqs, power_spectrum)):
        if f <= 0:
            continue
        # ✏️ TODO: compute midi = 12 * log2(f/440) + 69
        midi = 0.0
        # ✏️ TODO: pc = round(midi) mod 12, add into chroma[pc]
        pass

    # ✏️ TODO: normalize (divide by the sum; if sum==0 leave all zeros)
    return chroma

In [ ]:
# Check: chroma[9] should be highest for a pure A4 signal
try:
    sr_c, n_fft_c = 22050, 2048
    t_c = np.arange(n_fft_c) / sr_c
    x_c = np.sin(2 * np.pi * 440 * t_c) * np.hanning(n_fft_c)
    pspec = np.abs(np.fft.rfft(x_c)) ** 2

    ch = chroma_vector(pspec, sr_c, n_fft_c)
    peak_pc = ch.argmax()
    assert peak_pc == 9, f'expected 9 (A), got {peak_pc}'
    assert abs(ch.sum() - 1.0) < 1e-6, f'should be L1-normalized (divided by the sum), sum={ch.sum():.6f}'
    assert ch[9] > 0.99, f'a pure A4 tone should put its energy in chroma[9], got {ch[9]:.4f}'
    print(f'✅ chroma_vector OK: peak pitch class = {peak_pc} (A), chroma[9] = {ch[9]:.4f}, sum = {ch.sum():.4f}')
except (NotImplementedError, TypeError, AttributeError) as e:
    print(f'⏸ chroma_vector not implemented yet: {e}\nFollow the 4-step route to finish the TODO above.')

## 5. ✏️ Implement `rms_envelope(x, frame_len=2048, hop=512)`

**Signature**:
```python
def rms_envelope(x: np.ndarray, frame_len: int = 2048, hop: int = 512) -> np.ndarray:
    # x: 1D audio array
    # returns: (n_frames,) float, the RMS value of each frame
```

**3 steps to implement**:

| Step | Operation | Details |
|---|---|---|
| 1 | Framing: `n_frames = 1 + (len(x) - frame_len) // hop` | frame i starts at `i * hop` |
| 2 | Compute the RMS of each frame | `sqrt(mean(frame^2))` |
| 3 | Return shape=(n_frames,) | use list append or a preallocated array |

**Acceptance criteria**:
- All-zero signal → RMS all 0.0
- All-ones signal (length=frame_len) → RMS=1.0
- Time-domain frame-count formula: `n_frames = 1 + (N - frame_len) // hop`

In [ ]:
def rms_envelope(x, frame_len=2048, hop=512):
    """
    Compute the frame-by-frame RMS energy envelope of an audio signal.

    Parameters
    ----------
    x : np.ndarray, shape (N,)  mono audio
    frame_len : int  frame length (in samples)
    hop : int  hop size

    Returns
    -------
    rms : np.ndarray, shape (n_frames,)
    """
    # ✏️ TODO: compute n_frames = 1 + (len(x) - frame_len) // hop
    # ✏️ TODO: slice frames frames[i] = x[i*hop : i*hop+frame_len], shape = (n_frames, frame_len)
    # ✏️ TODO: return sqrt(mean(frame**2)) for each frame
    raise NotImplementedError("rms_envelope: please implement — follow the 3 steps above")

In [ ]:
# Check rms_envelope
try:
    # silence → RMS ≈ 0
    x_sil = np.zeros(10000)
    rms_sil = rms_envelope(x_sil)
    assert rms_sil.max() < 1e-10, f'RMS of silence should be about 0, got {rms_sil.max()}'

    # constant signal 0.5 → RMS ≈ 0.5
    x_const = np.ones(10000) * 0.5
    rms_const = rms_envelope(x_const)
    assert np.allclose(rms_const, 0.5, atol=1e-6), f'RMS of a constant signal should be 0.5, got {rms_const.mean():.4f}'

    print(f'✅ rms_envelope OK: silence max={rms_sil.max():.2e}, constant mean={rms_const.mean():.4f}')
except (NotImplementedError, TypeError, AttributeError) as e:
    print(f'⏸ rms_envelope not implemented yet: {e}\nFollow the steps above to finish the TODO in the rms_envelope cell.')

## 6. Parameter experiment: how hop affects the time resolution of RMS

Change the `hop` parameter and watch the time resolution and smoothness of the RMS envelope:

- `hop=128`: high time resolution, a jittery envelope, catches fast transients (< 6 ms)
- `hop=512` (default): balanced, a step of about 23 ms, good for beat detection
- `hop=2048`: low resolution, a very smooth envelope close to overall loudness; beat detail is lost

What to expect: the larger `hop`, the smoother the curve and the blurrier the beat peaks; the smaller `hop`, the finer the curve, but the computation grows in proportion.

In [ ]:
# Parameter experiment: RMS envelopes for different hops
sr_exp = 22050
bpm_exp = 120
beat_period = int(60 / bpm_exp * sr_exp)
x_exp = np.zeros(sr_exp * 4)
for bs in range(0, len(x_exp), beat_period):
    end = min(bs + 2205, len(x_exp))
    length = end - bs
    x_exp[bs:end] += np.exp(-np.arange(length) / 800) * 0.9

frame_len_exp = 2048
hops = [128, 512, 2048]
colors = ['tab:blue', 'tab:orange', 'tab:green']

try:
    rms_by_hop = [(h, c, rms_envelope(x_exp, frame_len=frame_len_exp, hop=h))
                  for h, c in zip(hops, colors)]
except (NotImplementedError, TypeError):
    print('⏸ rms_envelope not implemented yet, skipping the parameter experiment; finish the TODO above and re-run this cell.')
else:
    plt.figure(figsize=(11, 4))
    for h, c, rms_e in rms_by_hop:
        t_e = np.arange(len(rms_e)) * h / sr_exp
        plt.plot(t_e, rms_e, label=f'hop={h}', color=c, alpha=0.8)

    plt.xlabel('Time (s)')
    plt.ylabel('RMS')
    plt.title('RMS envelopes for different hops (simulated 120 BPM drum hits)')
    plt.legend()
    plt.tight_layout()
    plt.show()
    print('Look: hop=128 is the most detailed, hop=2048 the smoothest; beat peaks are sharper with a small hop.')

## 7. ✏️ Implement `zero_crossing_rate(x, frame_len=2048, hop=512)`

**Signature**:
```python
def zero_crossing_rate(x: np.ndarray, frame_len: int = 2048, hop: int = 512) -> np.ndarray:
    # returns: (n_frames,) float, the zero-crossing rate of each frame (between 0 and 1)
```

**Definition of zero-crossing rate**:
```
ZCR[i] = (1 / frame_len) × #{n: sign(x[n]) ≠ sign(x[n+1]), n ∈ frame i}
```

**Implementation**:
```python
# for each frame:
zcr = np.mean(np.abs(np.diff(np.sign(frame))) / 2)
# np.sign(frame): the sign of each sample (-1, 0, 1)
# np.diff(...): difference of neighboring signs; nonzero = a zero crossing
# /2: normalizes to [0,1]
```

**Acceptance criteria**:
- Pure tone (440 Hz sine wave, 22050 Hz sample rate): ZCR ≈ 2×440/22050 ≈ 0.04
- All-zero signal: ZCR=0.0
- White noise (random signs): ZCR close to 0.5

In [ ]:
def zero_crossing_rate(x, frame_len=2048, hop=512):
    """
    Compute the frame-by-frame zero-crossing rate of an audio signal.

    Parameters
    ----------
    x : np.ndarray, shape (N,)
    frame_len : int
    hop : int

    Returns
    -------
    zcr : np.ndarray, shape (n_frames,)  zero-crossing rate per frame, range [0, 1] (white noise ≈ 0.5)
    """
    # ✏️ TODO: 1. compute n_frames (same formula as rms_envelope)
    # ✏️ TODO: 2. slice the frames
    # ✏️ TODO: 3. for each frame, take np.mean(np.abs(np.diff(np.sign(frame))) / 2)
    raise NotImplementedError("zero_crossing_rate: please implement")


In [ ]:
# Check zero_crossing_rate
try:
    sr_z = 22050
    # pure A4 sine wave: ZCR ≈ 2 * 440 / 22050 ≈ 0.0399
    t_z = np.arange(sr_z) / sr_z
    x_sine = np.sin(2 * np.pi * 440 * t_z)
    zcr_sine = zero_crossing_rate(x_sine, frame_len=2048, hop=512)
    expected_zcr = 2 * 440 / sr_z   # ≈ 0.0399
    assert np.allclose(zcr_sine.mean(), expected_zcr, atol=0.005), \
        f'sine ZCR should be about {expected_zcr:.4f}, got {zcr_sine.mean():.4f}'

    # white noise: ZCR ≈ 0.5 (random signs, crossings are very frequent)
    rng = np.random.default_rng(42)
    x_noise = rng.standard_normal(sr_z)
    zcr_noise = zero_crossing_rate(x_noise, frame_len=2048, hop=512)
    assert zcr_noise.mean() > 0.45, f'white-noise ZCR should be > 0.45, got {zcr_noise.mean():.4f}'

    print(f'✅ zero_crossing_rate OK: sine ZCR mean={zcr_sine.mean():.4f} (expected ≈{expected_zcr:.4f}), '
          f'noise ZCR mean={zcr_noise.mean():.4f} (expected ≈0.5)')
except (NotImplementedError, TypeError, AttributeError) as e:
    print(f'⏸ zero_crossing_rate not implemented yet: {e}\nFollow the steps above to finish the stub.')

## Wrap-up

In this section you implemented `chroma_vector` (a 12-dimensional key fingerprint, returned as a normalized ndarray) and `rms_envelope` (frame-by-frame root mean square, returned as a loudness time series), and saw a demo of autocorrelation beat estimation (locating the BPM) — together these form the core of the `aurora.music` feature-extraction pipeline. The next section (**L78**) implements beat tracking from scratch: compute an onset envelope, estimate the BPM with autocorrelation, and check its accuracy on a synthetic drum signal.

## ✏️ Whiteboard challenge: music features by hand (target: 10 minutes)

Cover the screen and work it out on paper:

**Question 1**: What frequency does k=5 correspond to in the power spectrum? (sr=22050, n_fft=2048)
(f = k×sr/n_fft = 5×22050/2048 ≈ 53.8 Hz)

**Question 2**: What chroma value does 440 Hz (A4) map to? Show the derivation.
(c = round(12×log2(440/440)+69) % 12 = 69 % 12 = 9; A = chroma 9. 69 = 5×12+9 is the MIDI number of A4)

**Question 3**: With sr=22050, frame_len=2048, hop=512 and a signal of length N=44100, how many frames are there?
(n_frames = 1+(44100-2048)//512 = 1+42052//512 = 1+82 = 83 frames)

**Question 4**: Why is the ZCR of a pure tone ≈ 2×f/sr?
(A sine wave of frequency f crosses zero 2f times per second (twice per cycle); divide by the sample rate sr to get crossings per sample)

**Question 5**: Why does chroma_vector normalize by "dividing by the sum" rather than "dividing by the maximum"?
(Sum normalization turns the chroma vector into an "energy distribution" across the 12 pitch classes that adds up to 1, so clips of any loudness or length end up on the same scale and can be compared directly; dividing by the maximum keeps the ratios too, but ties the scale to the single strongest pitch class, so the vector no longer reads as a distribution)

When you've finished the derivations, run the cell below to check.

In [ ]:
# ✏️ Answer-check cell
import numpy as np

sr_test = 22050
n_fft_test = 2048
hop_test = 512
frame_len_test = 2048

# Q1: frequency of an FFT bin
k5_freq = 5 * sr_test / n_fft_test
expected_k5 = 53.8  # approximate
assert abs(k5_freq - expected_k5) < 0.2, f"k=5 frequency={k5_freq:.2f}, expected ≈{expected_k5}"
print(f"Q1 ✅  k=5: f={k5_freq:.2f} Hz (≈{expected_k5} Hz)")

# Q2: chroma of A4
import math
freq_a4 = 440.0
c_a4 = round(12 * math.log2(freq_a4 / 440) + 69) % 12
assert c_a4 == 9, f"440Hz→chroma should be 9, got {c_a4}"
print(f"Q2 ✅  440Hz(A4)→chroma=round(12×log2(1)+69)%12=69%12={c_a4} (A=9, 69=5×12+9)")

# Q3: frame-count formula
N_sig = 44100
n_frames_calc = 1 + (N_sig - frame_len_test) // hop_test
assert n_frames_calc == 83, f"frame count should be 83, got {n_frames_calc}"
print(f"Q3 ✅  N={N_sig},frame={frame_len_test},hop={hop_test}→n_frames=1+({N_sig}-{frame_len_test})//{hop_test}={n_frames_calc}")

# Q4: check ZCR
try:
    t_zcr = np.arange(frame_len_test) / sr_test
    f_zcr = 440.0
    x_sine = np.sin(2 * np.pi * f_zcr * t_zcr)
    zcr_val = zero_crossing_rate(x_sine, frame_len=frame_len_test, hop=frame_len_test)
    expected_zcr = 2 * f_zcr / sr_test
    assert abs(zcr_val[0] - expected_zcr) < 0.005, f"ZCR should be ≈{expected_zcr:.4f}, got {zcr_val[0]:.4f}"
    print(f"Q4 ✅  pure tone {f_zcr}Hz: ZCR={zcr_val[0]:.4f}≈2×{f_zcr}/{sr_test}={expected_zcr:.4f}")
except (NotImplementedError, TypeError):
    expected_zcr = 2 * 440 / sr_test
    print(f"Q4 ✅  ZCR in theory: 2×f/sr=2×440/{sr_test}≈{expected_zcr:.4f} (will be verified once implemented)")

# Q5: chroma normalization (check the properties of sum normalization)
try:
    t_chk = np.arange(n_fft_test) / sr_test
    x_a4 = np.sin(2 * np.pi * 440 * t_chk)
    ps_a4 = np.abs(np.fft.rfft(x_a4)) ** 2
    ch = chroma_vector(ps_a4, sr_test, n_fft_test)
    assert abs(ch.sum() - 1.0) < 1e-6 or ch.sum() == 0, f"chroma should sum to ≈1.0, got {ch.sum():.6f}"
    assert ch[9] == ch.max(), f"chroma[9] should be the maximum for A4, actual argmax={ch.argmax()}"
    print(f"Q5 ✅  chroma sum normalization: sum={ch.sum():.4f}, maximum at chroma[{ch.argmax()}]=A={ch[9]:.4f}")
except (NotImplementedError, TypeError):
    print(f"Q5 ✅  sum normalization → an energy distribution, comparable across clips (will be verified once implemented)")

print("\n🎉 Music-features whiteboard challenge passed!")

In [ ]:
# ✏️ Self-review for this lesson
l77_review = {
    "chroma_logic_understood":  None,  # understand chroma = FFT power summed by pitch class? True/False
    "chroma_vector_impl":       None,  # chroma_vector() implemented correctly, pure A4 → chroma[9] is the max? True/False
    "rms_envelope_impl":        None,  # rms_envelope() implemented correctly, all zeros → 0, all ones → 1? True/False
    "zcr_impl":                 None,  # zero_crossing_rate() implemented correctly, ZCR≈2f/sr? True/False
    "whiteboard_passed":        None,  # all 5 whiteboard questions worked by hand? True/False
}

unfilled = [k for k, v in l77_review.items() if v is None]
assert not unfilled, f'not filled in yet: {unfilled}'
weak = [k for k, v in l77_review.items() if v is False]
if weak:
    print(f'⚠️  needs more work: {weak}')
else:
    print('✅ L77 complete! On to L78: beat tracking from scratch')

---

→ **Next lesson** · [L78 · Beat tracking from scratch](L78_beat_tracking.ipynb)

> Next lesson covers **beat tracking from scratch**: the onset envelope, autocorrelation and BPM estimation.